# KVCompass Extended Features Testing

This notebook tests the expanded quantization options and token-limited scenarios using the KVCompass evaluation scripts.

## New Features:
1. **Additional Quantization Methods**: 2-bit, 8-bit, grouped quantization, and HQQ backend
2. **Token-Limited Scenario**: Simulates API constraints with total token limits

In [ ]:
# Clone repo and enter project
# !git clone https://github.com/AarnavSawant/KVCompass.git
%cd KVCompass

# Optional: verify GPU
!nvidia-smi

# Install dependencies and setup
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!pip install -e .

In [ ]:
from google.colab import userdata
import os
userdataname = "HF_TOKEN"
userdatavalue = userdata.get(userdataname)
os.environ["HF_TOKEN"] = userdatavalue

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
# Test token-limited scenario with all methods

# Test with SnapKV
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method snapkv \
  --torch-dtype bfloat16 \
  --max-cases 2 \
  --fraction 0.02 \
  --output results/test_token_limited_snap.csv \
  --run-name test_extended

In [ ]:
# Test with Expected Attention
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method expected_attention \
  --torch-dtype bfloat16 \
  --max-cases 2 \
  --fraction 0.02 \
  --output results/test_token_limited_expected_attention.csv \
  --run-name test_extended

In [ ]:
# Test with AdaKVPress
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method adakv_expected \
  --torch-dtype bfloat16 \
  --max-cases 2 \
  --fraction 0.02 \
  --output results/test_token_limited_adakv_expected.csv \
  --run-name test_extended

In [ ]:
# Test with Per Layer Compression Press
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method layerwise \
  --torch-dtype bfloat16 \
  --max-cases 2 \
  --fraction 0.02 \
  --output results/test_token_limited_layerwise.csv \
  --run-name test_extended

In [ ]:
# Test with 4-bit quantization
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method quant_4bit \
  --torch-dtype bfloat16 \
  --max-cases 2 \
  --fraction 0.02 \
  --output results/test_token_limited_4bit.csv \
  --run-name test_extended

In [ ]:
# Test with grouped 4-bit quantization
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method quant_4bit_grouped \
  --device cpu \
  --torch-dtype float32 \
  --max-cases 2 \
  --output results/test_token_limited_4bit_grouped.csv \
  --run-name test_extended

In [ ]:
# Compare with baseline (no compression)
!python scripts/run_kvpress_eval.py \
  --dataset ruler \
  --data-dir 4096 \
  --model Qwen/Qwen2.5-1.5B-Instruct \
  --scenario token_limited \
  --method no_compression \
  --device cpu \
  --torch-dtype float32 \
  --max-cases 2 \
  --output results/test_token_limited_baseline.csv \
  --run-name test_extended

In [ ]:
# Load and analyze results
import pandas as pd
import glob

# Find all test result files
result_files = glob.glob("results/test_token_limited_*.csv")
print(f"Found {len(result_files)} result files")

# Load and combine results
all_results = []
for file in result_files:
    try:
        df = pd.read_csv(file)
        all_results.append(df)
        print(f"Loaded {len(df)} rows from {file}")
    except Exception as e:
        print(f"Error loading {file}: {e}")

if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)
    print(f"\nCombined results shape: {combined_df.shape}")
    print("\nColumns:", list(combined_df.columns))
    
    # Show summary by method
    summary = combined_df.groupby('method').agg({
        'quality_score': 'mean',
        'latency_seconds': 'mean',
        'compression_ratio': 'mean',
        'input_tokens': 'mean',
        'output_tokens': 'mean'
    }).round(4)
    print("\nSummary by method:")
    print(summary)
else:
    print("No results loaded")

In [ ]:
# Configuration verification
from kvpress_eval.config import get_method_configs, get_scenario_configs

print("=== Available Methods ===")
methods = get_method_configs('configs/methods.yaml')
for method in methods:
    if 'quantized_cache' in method.get('kind', ''):
        print(f"- {method['name']}: {method['nbits']}-bit, backend: {method.get('backend', 'quanto')}")

print("\n=== Token-Limited Scenario ===")
scenarios = get_scenario_configs('configs/scenarios.yaml')
token_scenario = next((s for s in scenarios if s['name'] == 'token_limited'), None)
if token_scenario:
    print(f"Max total tokens: {token_scenario['max_total_tokens']}")
    print(f"Context lengths: {token_scenario['context_lengths']}")
    print(f"Max new tokens: {token_scenario['max_new_tokens']}")
    print("Note: Context lengths are automatically adjusted to stay within total token limits")

## Summary

This notebook demonstrates testing the extended KVCompass features:

### New Quantization Methods Tested:
- **quant_4bit**: Standard 4-bit quantization
- **quant_4bit_grouped**: 4-bit with group size 64 for better quality
- **quant_4bit_hqq**: 4-bit using HQQ backend

### Token-Limited Scenario:
- Simulates API constraints with `max_total_tokens: 4096, 2048`
- Context lengths automatically adjusted to stay within limits
- Tests compression effectiveness under realistic constraints

The results can then be analyzed using the existing plotting and aggregation scripts in the `scripts/` directory.